# IEEE-CIS Fraud Detection — RandomForest
## Sections: Cleaning | Feature Engineering | Feature Selection | Training

## 0. Installation & Setup

In [1]:
!pip install dagshub mlflow scikit-learn pandas numpy -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 879.5/

In [2]:
import os, gc, warnings
import numpy as np
import pandas as pd
import mlflow, mlflow.sklearn
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from mlflow.models.signature import infer_signature
from scipy.stats import randint

def reduce_mem_usage(df):
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object:
            c_min = df[col].min(); c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min   and c_max < np.iinfo(np.int8).max:   df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max: df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max: df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max: df[col] = df[col].astype(np.float32)
    return df

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ['MLFLOW_TRACKING_USERNAME'] = 'dgrig23'
os.environ['MLFLOW_TRACKING_PASSWORD'] = secrets.get_secret('DAGSHUB_TOKEN')

REPO = 'dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning'
mlflow.set_tracking_uri(f'https://dagshub.com/{REPO}.mlflow')
mlflow.set_experiment('RandomForest_Training')
EXP_PREFIX = 'RF'
BASE = '/kaggle/input/competitions/ieee-fraud-detection/'
print('Ready.')


Ready.


## 1. Cleaning

In [3]:
train_trx = reduce_mem_usage(pd.read_csv(BASE + 'train_transaction.csv'))
train_idn = reduce_mem_usage(pd.read_csv(BASE + 'train_identity.csv'))
train_idn.columns = train_idn.columns.str.replace('-', '_')
train = train_trx.merge(train_idn, on='TransactionID', how='left')
del train_trx, train_idn; gc.collect()
print(f'Train shape: {train.shape}')

with mlflow.start_run(run_name=f'{EXP_PREFIX}_Cleaning'):
    HIGH_MISS = 0.9
    miss = train.isnull().mean()
    high_miss_cols = miss[miss > HIGH_MISS].index.tolist()
    train.drop(columns=high_miss_cols + ['TransactionID'], inplace=True, errors='ignore')
    y = train.pop('isFraud').copy()
    fraud_rate = y.mean()
    mlflow.log_params({'high_miss_threshold': HIGH_MISS, 'cols_dropped': len(high_miss_cols),
                       'class_weight': 'balanced', 'note': 'RF_handles_imbalance_natively'})
    mlflow.log_metrics({'fraud_rate': round(float(fraud_rate), 4), 'cols_after': train.shape[1]})
    print(f'Fraud rate: {fraud_rate:.4f} | Columns: {train.shape[1]}')


Train shape: (590540, 434)
Fraud rate: 0.0350 | Columns: 420
🏃 View run RF_Cleaning at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/4/runs/4a7bc81afaef4f4ea3b1879667269d48
🧪 View experiment at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/4


## 2. Feature Engineering

In [4]:
with mlflow.start_run(run_name=f'{EXP_PREFIX}_Feature_Engineering'):
    cols_before = train.shape[1]

    if 'TransactionDT' in train.columns:
        train['hour']      = ((train['TransactionDT'] / 3600)        % 24).astype(np.float32)
        train['dayofweek'] = ((train['TransactionDT'] / (3600*24))   %  7).astype(np.float32)
        train['week']      = ((train['TransactionDT'] / (3600*24*7)) % 52).astype(np.float32)
        train.drop(columns=['TransactionDT'], inplace=True)

    for col in ['P_emaildomain', 'R_emaildomain']:
        if col in train.columns:
            train[col+'_suffix'] = train[col].str.split('.').str[-1].fillna('unknown')
            train[col+'_domain'] = train[col].str.split('.').str[0].fillna('unknown')
    if 'P_emaildomain' in train.columns and 'R_emaildomain' in train.columns:
        train['email_match'] = (train['P_emaildomain'] == train['R_emaildomain']).astype(np.int8)

    if 'TransactionAmt' in train.columns:
        train['TransactionAmt_log']   = np.log1p(train['TransactionAmt']).astype(np.float32)
        train['TransactionAmt_cents'] = (train['TransactionAmt'] % 1).astype(np.float32)
        train['amt_is_round']         = (train['TransactionAmt'] % 1   == 0).astype(np.int8)
        train['amt_is_round_100']     = (train['TransactionAmt'] % 100 == 0).astype(np.int8)

    for g in ['card1', 'card4', 'addr1']:
        if g in train.columns:
            agg = train.groupby(g)['TransactionAmt'].agg(['mean','std','max'])
            agg.columns = [f'{g}_amt_mean', f'{g}_amt_std', f'{g}_amt_max']
            train = train.join(agg, on=g)
            train[f'{g}_amt_std'] = train[f'{g}_amt_std'].fillna(0)
            train[f'{g}_amt_zscore'] = ((train['TransactionAmt'] - train[f'{g}_amt_mean']) /
                                        train[f'{g}_amt_std'].replace(0,1)).clip(-5,5).astype(np.float32)

    uid_parts = [c for c in ['card1','card2','addr1','P_emaildomain'] if c in train.columns]
    train['user_id'] = train[uid_parts].fillna(-1).astype(str).agg('_'.join, axis=1)
    u = train.groupby('user_id')['TransactionAmt'].agg(['count','mean','std','max'])
    u.columns = ['uid_count','uid_mean','uid_std','uid_max']
    train = train.join(u, on='user_id')
    train['uid_std'] = train['uid_std'].fillna(0)
    train['user_amt_zscore'] = ((train['TransactionAmt'] - train['uid_mean']) /
                                train['uid_std'].replace(0,1)).clip(-5,5).astype(np.float32)
    train['user_count_log']  = np.log1p(train['uid_count']).astype(np.float32)
    train['user_amt_vs_max'] = (train['TransactionAmt'] / train['uid_max']).clip(0,1).astype(np.float32)
    train.drop(columns=['user_id'], inplace=True)

    train = reduce_mem_usage(train)
    mlflow.log_params({'time':'hour,dow,week','email':'suffix,domain,match',
                       'amount':'log,cents,round,round100','aggs':'card1,card4,addr1,user_id'})
    mlflow.log_metrics({'new_features': train.shape[1]-cols_before, 'total_features': train.shape[1]})
    print(f'Features: {cols_before} → {train.shape[1]}')


Features: 420 → 450
🏃 View run RF_Feature_Engineering at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/4/runs/676654988fbe4fe4808ce42ac75e56b9
🧪 View experiment at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/4


In [5]:
cat_cols = train.select_dtypes(include='object').columns.tolist()
le_store = {}
for col in cat_cols:
    train[col] = train[col].fillna('unknown').astype(str)
    le = LabelEncoder()
    le.fit(list(train[col].unique()) + ['unknown'])
    le_store[col] = le
    train[col] = le.transform(train[col]).astype(np.int32)

num_cols = train.select_dtypes(include=[np.number]).columns.tolist()
medians  = train[num_cols].median()
train[num_cols] = train[num_cols].fillna(medians).astype(np.float32)
print(f'Missing: {train.isnull().sum().sum()} | Shape: {train.shape}')


Missing: 0 | Shape: (590540, 450)


## 3. Feature Selection

In [6]:
X_tr, X_va, y_tr, y_va = train_test_split(train, y, test_size=0.2, stratify=y, random_state=42)
X_sub, _, y_sub, _ = train_test_split(X_tr, y_tr, train_size=0.05, stratify=y_tr, random_state=42)

def quick_eval(features):
    m = RandomForestClassifier(n_estimators=30, max_depth=10, n_jobs=-1,
                               class_weight='balanced', random_state=42)
    m.fit(X_sub[features], y_sub)
    return roc_auc_score(y_va, m.predict_proba(X_va[features])[:,1])

with mlflow.start_run(run_name=f'{EXP_PREFIX}_Feature_Selection'):
    corr = train.corrwith(y).abs()
    corr_features = corr[corr >= 0.005].index.tolist()
    auc_corr = quick_eval(corr_features)
    print(f'Strategy A – Correlation filter ({len(corr_features)} features): AUC = {auc_corr:.5f}')

    imp_clf = RandomForestClassifier(n_estimators=50, max_depth=10, n_jobs=-1,
                                     class_weight='balanced', random_state=42)
    imp_clf.fit(X_sub[corr_features], y_sub)
    imp = pd.Series(imp_clf.feature_importances_, index=corr_features)
    top_features = imp.nlargest(60).index.tolist()
    auc_top = quick_eval(top_features)
    print(f'Strategy B – RF Importance top-60 ({len(top_features)} features): AUC = {auc_top:.5f}')

    best_strategy, best_auc, final_features = max(
        [('correlation_0.005', auc_corr, corr_features),
         ('rf_importance_top60', auc_top, top_features)],
        key=lambda x: x[1]
    )
    mlflow.log_params({'strategy_A': 'correlation_0.005',
                       'strategy_B': 'rf_importance_top60',
                       'selected':   best_strategy,
                       'n_final':    len(final_features)})
    mlflow.log_metrics({'auc_corr':   round(auc_corr, 5),
                        'auc_top60':  round(auc_top,  5),
                        'best_auc':   round(best_auc,  5)})
    print(f'→ Best: {best_strategy} | AUC: {best_auc:.5f} | Features: {len(final_features)}')


Strategy A – Correlation filter (360 features): AUC = 0.86117
Strategy B – RF Importance top-60 (60 features): AUC = 0.85933
→ Best: correlation_0.005 | AUC: 0.86117 | Features: 360
🏃 View run RF_Feature_Selection at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/4/runs/cb8b0b6e8fb8498791012471261d1c7c
🧪 View experiment at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/4


## 4. Training — RandomForest

In [7]:
X = train[final_features].copy()
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# (a) Underfit 
with mlflow.start_run(run_name=f'{EXP_PREFIX}_Underfit_Config'):
    m = RandomForestClassifier(n_estimators=10, max_depth=3, n_jobs=-1,
                               class_weight='balanced', random_state=42)
    m.fit(X_tr, y_tr)
    tr_auc = roc_auc_score(y_tr, m.predict_proba(X_tr)[:,1])
    va_auc = roc_auc_score(y_va, m.predict_proba(X_va)[:,1])
    mlflow.log_params({'n_estimators':10,'max_depth':3,
                       'note':'very_shallow_high_bias_underfit'})
    mlflow.log_metrics({'train_auc':round(tr_auc,5),'val_auc':round(va_auc,5),
                        'overfit_gap':round(tr_auc-va_auc,5)})
    print(f'Underfit — Train: {tr_auc:.5f} | Val: {va_auc:.5f}')

# (b) Overfit
with mlflow.start_run(run_name=f'{EXP_PREFIX}_Overfit_Config'):
    tiny_idx = np.random.RandomState(42).choice(len(X_tr),
                                                size=int(0.05*len(X_tr)), replace=False)
    X_tiny = X_tr.iloc[tiny_idx]; y_tiny = y_tr.iloc[tiny_idx]
    m = RandomForestClassifier(n_estimators=200, max_depth=None, min_samples_leaf=1,
                               n_jobs=-1, class_weight='balanced', random_state=42)
    m.fit(X_tiny, y_tiny)
    tr_auc = roc_auc_score(y_tiny, m.predict_proba(X_tiny)[:,1])
    va_auc = roc_auc_score(y_va,   m.predict_proba(X_va)[:,1])
    mlflow.log_params({'n_estimators':200,'max_depth':'None','min_samples_leaf':1,
                       'train_size':'5% of 80%','note':'unlimited_depth_memorises_training_data'})
    mlflow.log_metrics({'train_auc':round(tr_auc,5),'val_auc':round(va_auc,5),
                        'overfit_gap':round(tr_auc-va_auc,5)})
    print(f'Overfit  — Train: {tr_auc:.5f} | Val: {va_auc:.5f} | Gap: {tr_auc-va_auc:.5f}')


Underfit — Train: 0.80548 | Val: 0.80895
🏃 View run RF_Underfit_Config at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/4/runs/d99df5368899485d8f4742b88711e641
🧪 View experiment at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/4
Overfit  — Train: 1.00000 | Val: 0.88965 | Gap: 0.11035
🏃 View run RF_Overfit_Config at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/4/runs/09687924f7f64c2d9fdc2dcc525a498c
🧪 View experiment at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/4


In [8]:
# (c) RandomizedSearch
with mlflow.start_run(run_name=f'{EXP_PREFIX}_RandomizedSearch') as run_rs:
    X_rs = X.sample(frac=0.10, random_state=42)
    y_rs = y.loc[X_rs.index]

    param_dist = {
        'n_estimators':      [100, 150, 200],
        'max_depth':         [8, 12, 16, None],
        'min_samples_leaf':  randint(1, 20),
        'max_features':      ['sqrt', 'log2', 0.3],
        'min_samples_split': randint(2, 20),
    }

    base_clf = RandomForestClassifier(class_weight='balanced', n_jobs=-1, random_state=42)
    skf2 = StratifiedKFold(2, shuffle=True, random_state=42)
    rs = RandomizedSearchCV(base_clf, param_dist, n_iter=5, cv=skf2,
                            scoring='roc_auc', n_jobs=1,
                            return_train_score=True, random_state=42, verbose=1)
    rs.fit(X_rs, y_rs)

    for i, params in enumerate(rs.cv_results_['params']):
        label = f"n{params['n_estimators']}_d{params['max_depth']}_mf{params['max_features']}"
        with mlflow.start_run(run_name=f'RF_RS_{label}', nested=True):
            mlflow.log_params({str(k): str(v) for k, v in params.items()})
            mlflow.log_metrics({
                'cv_auc_mean':    round(float(rs.cv_results_['mean_test_score'][i]),  5),
                'cv_auc_std':     round(float(rs.cv_results_['std_test_score'][i]),   5),
                'train_auc_mean': round(float(rs.cv_results_['mean_train_score'][i]), 5),
                'overfit_gap':    round(float(rs.cv_results_['mean_train_score'][i] -
                                             rs.cv_results_['mean_test_score'][i]),  5),
            })

    best_params = rs.best_params_
    mlflow.log_params({'best_'+k: str(v) for k, v in best_params.items()})
    mlflow.log_metric('best_cv_auc', round(rs.best_score_, 5))
    print(f'Best CV AUC: {rs.best_score_:.5f} | Params: {best_params}')


Fitting 2 folds for each of 5 candidates, totalling 10 fits
🏃 View run RF_RS_n100_d16_mfsqrt at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/4/runs/905a6dbf817e4b30a435e95961616992
🧪 View experiment at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/4
🏃 View run RF_RS_n200_d8_mf0.3 at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/4/runs/bec94cc477544b589c25968278155744
🧪 View experiment at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/4
🏃 View run RF_RS_n200_dNone_mfsqrt at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/4/runs/f73c6a3413274167baf03b9d8025b638
🧪 View experiment at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/4
🏃 View run RF_RS_n150_d12_mfsqrt at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud

In [9]:
# (d) Final Pipeline 
class ColumnSelector(BaseEstimator, TransformerMixin):
    """Keeps only the selected feature columns."""
    def __init__(self, cols): self.cols = cols
    def fit(self, X, y=None): return self
    def transform(self, X):
        present = [c for c in self.cols if c in X.columns]
        return X[present]

class FraudPreprocessorRF(BaseEstimator, TransformerMixin):
    def __init__(self, miss_thresh=0.9):
        self.miss_thresh     = miss_thresh
        self.high_miss_cols_ = []
        self.le_store_       = {}
        self.medians_        = None
        self.features_       = None
        self.card_stats_     = {}
        self.user_stats_     = None

    def fit(self, X, y=None):
        X = X.copy(); X.columns = X.columns.str.replace('-','_')
        for c in ['TransactionID','isFraud']:
            if c in X.columns: X.drop(columns=[c], inplace=True)
        self.high_miss_cols_ = X.columns[X.isnull().mean() > self.miss_thresh].tolist()
        X.drop(columns=self.high_miss_cols_, inplace=True, errors='ignore')
        for g in ['card1','card4','addr1']:
            if g in X.columns:
                agg = X.groupby(g)['TransactionAmt'].agg(['mean','std','max'])
                agg.columns = [f'{g}_amt_mean',f'{g}_amt_std',f'{g}_amt_max']
                self.card_stats_[g] = agg
        uid_parts = [c for c in ['card1','card2','addr1','P_emaildomain'] if c in X.columns]
        X['_uid'] = X[uid_parts].fillna(-1).astype(str).agg('_'.join, axis=1)
        u = X.groupby('_uid')['TransactionAmt'].agg(['count','mean','std','max'])
        u.columns = ['uid_count','uid_mean','uid_std','uid_max']
        self.user_stats_ = u
        X.drop(columns=['_uid'], inplace=True)
        X = self._engineer(X)
        for col in X.select_dtypes(include='object').columns:
            le = LabelEncoder(); vals = X[col].fillna('unknown').astype(str)
            le.fit(list(vals.unique()) + ['unknown'])
            self.le_store_[col] = le; X[col] = le.transform(vals).astype(np.int32)
        num = X.select_dtypes(include=[np.number]).columns.tolist()
        self.medians_ = X[num].median()
        X[num] = X[num].fillna(self.medians_).astype(np.float32)
        self.features_ = X.columns.tolist()
        return self

    def transform(self, X):
        X = X.copy(); X.columns = X.columns.str.replace('-','_')
        for c in ['TransactionID','isFraud']:
            if c in X.columns: X.drop(columns=[c], inplace=True)
        X.drop(columns=self.high_miss_cols_, inplace=True, errors='ignore')
        X = self._engineer(X)
        for col in X.select_dtypes(include='object').columns:
            if col in self.le_store_:
                known = set(self.le_store_[col].classes_)
                X[col] = X[col].fillna('unknown').astype(str)
                X.loc[~X[col].isin(known), col] = 'unknown'
                X[col] = self.le_store_[col].transform(X[col]).astype(np.int32)
        num = X.select_dtypes(include=[np.number]).columns.tolist()
        X[num] = X[num].fillna(self.medians_).astype(np.float32)
        for f in self.features_:
            if f not in X.columns: X[f] = 0
        return X[self.features_]

    def _engineer(self, df):
        if 'TransactionDT' in df.columns:
            df['hour']      = ((df['TransactionDT']/3600)        % 24).astype(np.float32)
            df['dayofweek'] = ((df['TransactionDT']/(3600*24))   %  7).astype(np.float32)
            df['week']      = ((df['TransactionDT']/(3600*24*7)) % 52).astype(np.float32)
            df.drop(columns=['TransactionDT'], inplace=True)
        for col in ['P_emaildomain','R_emaildomain']:
            if col in df.columns:
                df[col+'_suffix'] = df[col].str.split('.').str[-1].fillna('unknown')
                df[col+'_domain'] = df[col].str.split('.').str[0].fillna('unknown')
        if 'P_emaildomain' in df.columns and 'R_emaildomain' in df.columns:
            df['email_match'] = (df['P_emaildomain'] == df['R_emaildomain']).astype(np.int8)
        if 'TransactionAmt' in df.columns:
            df['TransactionAmt_log']   = np.log1p(df['TransactionAmt']).astype(np.float32)
            df['TransactionAmt_cents'] = (df['TransactionAmt'] % 1).astype(np.float32)
            df['amt_is_round']         = (df['TransactionAmt'] % 1   == 0).astype(np.int8)
            df['amt_is_round_100']     = (df['TransactionAmt'] % 100 == 0).astype(np.int8)
        for g, stats in self.card_stats_.items():
            if g in df.columns:
                df = df.join(stats, on=g)
                df[f'{g}_amt_std'] = df[f'{g}_amt_std'].fillna(0)
                df[f'{g}_amt_zscore'] = ((df['TransactionAmt'] - df[f'{g}_amt_mean']) /
                                         df[f'{g}_amt_std'].replace(0,1)).clip(-5,5).astype(np.float32)
        if self.user_stats_ is not None:
            uid_parts = [c for c in ['card1','card2','addr1','P_emaildomain'] if c in df.columns]
            df['_uid'] = df[uid_parts].fillna(-1).astype(str).agg('_'.join, axis=1)
            df = df.join(self.user_stats_, on='_uid')
            df['uid_std'] = df['uid_std'].fillna(0)
            df['user_amt_zscore'] = ((df['TransactionAmt'] - df['uid_mean']) /
                                     df['uid_std'].replace(0,1)).clip(-5,5).astype(np.float32)
            df['user_count_log']  = np.log1p(df['uid_count'].fillna(1)).astype(np.float32)
            df['user_amt_vs_max'] = (df['TransactionAmt'] / df['uid_max'].replace(0,1)).clip(0,1).astype(np.float32)
            df.drop(columns=['_uid'], inplace=True)
        return df

print('Reloading raw data for final pipeline...')
raw_trx   = pd.read_csv(BASE + 'train_transaction.csv')
raw_idn   = pd.read_csv(BASE + 'train_identity.csv')
raw_idn.columns = raw_idn.columns.str.replace('-', '_')
raw_train = raw_trx.merge(raw_idn, on='TransactionID', how='left')
y_raw     = raw_train['isFraud'].copy()
del raw_trx, raw_idn; gc.collect()

final_clf_params = dict(class_weight='balanced', n_jobs=-1, random_state=42, **best_params)
final_pipeline = Pipeline([
    ('preprocessor', FraudPreprocessorRF(miss_thresh=0.9)),
    ('selector',     ColumnSelector(final_features)),
    ('classifier',   RandomForestClassifier(**final_clf_params)),
])

with mlflow.start_run(run_name=f'{EXP_PREFIX}_Final_Model') as run_final:
    mlflow.log_params({**{k: str(v) for k,v in final_clf_params.items()},
                       'miss_thresh': 0.9,
                       'feature_strategy': best_strategy,
                       'n_features': len(final_features)})

    raw_tr, raw_va, y_tr_raw, y_va_raw = train_test_split(
        raw_train, y_raw, test_size=0.2, stratify=y_raw, random_state=42)
    final_pipeline.fit(raw_tr, y_tr_raw)
    val_auc   = roc_auc_score(y_va_raw, final_pipeline.predict_proba(raw_va)[:,1])
    train_auc = roc_auc_score(y_tr_raw, final_pipeline.predict_proba(raw_tr)[:,1])

    mlflow.log_metrics({'train_auc':   round(float(train_auc), 5),
                        'val_auc':     round(float(val_auc),   5),
                        'overfit_gap': round(float(train_auc - val_auc), 5)})
    print(f'Train AUC: {train_auc:.5f} | Val AUC: {val_auc:.5f}')

    sample_in  = raw_train.head(5)
    sample_out = final_pipeline.predict_proba(sample_in)[:,1]
    sig        = infer_signature(sample_in, sample_out)

    mlflow.sklearn.log_model(
        final_pipeline,
        artifact_path='rf_fraud_pipeline',
        signature=sig,
        registered_model_name='RandomForest_FraudDetection',
    )
    print('Pipeline registered as RandomForest_FraudDetection v1')
    print(f'Run ID: {run_final.info.run_id}')


Reloading raw data for final pipeline...
Train AUC: 0.99962 | Val AUC: 0.95639


2026/05/07 11:02:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 11:02:44 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'RandomForest_FraudDetection' already exists. Creating a new version of this model...
2026/05/07 11:03:41 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: RandomForest_FraudDetection, version 4
Created version '4' of model 'RandomForest_FraudDetection'.


Pipeline registered as RandomForest_FraudDetection v1
Run ID: c5496256bbcd4841b6fb32de9571a23c
🏃 View run RF_Final_Model at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/4/runs/c5496256bbcd4841b6fb32de9571a23c
🧪 View experiment at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/4
